# Gemma SOKE Autoregressive Validation

Runs greedy autoregressive validation for Text-to-Motion and caption-output tasks. Text-to-Motion BLEU/ROUGE are scored separately for body, left hand, right hand, and combined streams; decoded motion is scored with MPJPE, PA-MPJPE, and DTW-MPJPE.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

LOCAL_DEFAULT = Path('/home/cem/tez/exp/new_data/04_soke_gemma3_270m_lora_lm')
COLAB_DEFAULT = Path('/content/drive/MyDrive/folder/COLAB/Tokenizer/04_soke_gemma3_270m_lora_lm')
BUNDLE_ROOT = Path(os.environ.get('SOKE_GEMMA_BUNDLE_ROOT', str(COLAB_DEFAULT if COLAB_DEFAULT.exists() else LOCAL_DEFAULT)))
if not (BUNDLE_ROOT / 'scripts' / 'validate_gemma_soke_autoreg.py').exists():
    BUNDLE_ROOT = Path.cwd()

ADAPTER_CANDIDATES = [
    BUNDLE_ROOT / 'outputs' / 'runs' / 'gemma3_270m_lora_soke_flat_lm' / 'lm_instruct' / 'last_adapter',
    Path('/mnt/y/Drive/My_Drive/folder/COLAB/Tokenizer/04_soke_gemma3_270m_lora_lm/outputs/runs/gemma3_270m_lora_soke_flat_lm/lm_instruct/last_adapter'),
    Path('/content/drive/MyDrive/folder/COLAB/Tokenizer/04_soke_gemma3_270m_lora_lm/outputs/runs/gemma3_270m_lora_soke_flat_lm/lm_instruct/last_adapter'),
]
default_adapter = next((p for p in ADAPTER_CANDIDATES if (p / 'adapter_config.json').exists()), ADAPTER_CANDIDATES[0])

BASE_MODEL = os.environ.get('SOKE_GEMMA_BASE_MODEL', 'google/gemma-3-270m')
ADAPTER = Path(os.environ.get('SOKE_GEMMA_VAL_ADAPTER', str(default_adapter)))
SPLIT = os.environ.get('SOKE_GEMMA_VAL_SPLIT', 'val')
MAX_ROWS = int(os.environ.get('SOKE_GEMMA_VAL_MAX_ROWS', '64'))
MAX_T2M_TASKS = int(os.environ.get('SOKE_GEMMA_VAL_MAX_T2M_TASKS', '0'))
MAX_CAPTION_TASKS = int(os.environ.get('SOKE_GEMMA_VAL_MAX_CAPTION_TASKS', '0'))
MAX_NEW_MOTION_TOKENS = int(os.environ.get('SOKE_GEMMA_VAL_MAX_NEW_MOTION_TOKENS', '384'))
MAX_NEW_TEXT_TOKENS = int(os.environ.get('SOKE_GEMMA_VAL_MAX_NEW_TEXT_TOKENS', '96'))
DTYPE = os.environ.get('SOKE_GEMMA_VAL_DTYPE', 'bf16')
ATTN_IMPLEMENTATION = os.environ.get('SOKE_GEMMA_VAL_ATTN_IMPLEMENTATION', 'sdpa')
NORM_SOURCE = os.environ.get('SOKE_GEMMA_VAL_NORM_SOURCE', 'official')
RUN_VALIDATION = os.environ.get('SOKE_GEMMA_RUN_VALIDATION', '1') == '1'
FORCE = os.environ.get('SOKE_GEMMA_VAL_FORCE', '0') == '1'

CODE_ROOT = Path(os.environ.get('SOKE_GEMMA_CODE_ROOT', str(BUNDLE_ROOT / 'outputs' / 'soke_motion_codes')))
INSTRUCTIONS_ROOT = Path(os.environ.get('SOKE_GEMMA_INSTRUCTIONS_ROOT', str(BUNDLE_ROOT / 'instructions')))
OUTPUT_ROOT = Path(os.environ.get('SOKE_GEMMA_VAL_OUTPUT_ROOT', str(BUNDLE_ROOT / '05_autoreg_validation')))
BODY_MODEL_ROOT = Path(os.environ.get('SMPLX_MODEL_ROOT', '/home/cem/tez/exp/body_models' if LOCAL_DEFAULT.exists() else '/content/body_models'))

print({
    'BUNDLE_ROOT': str(BUNDLE_ROOT),
    'CODE_ROOT': str(CODE_ROOT),
    'INSTRUCTIONS_ROOT': str(INSTRUCTIONS_ROOT),
    'OUTPUT_ROOT': str(OUTPUT_ROOT),
    'ADAPTER': str(ADAPTER),
    'BASE_MODEL': BASE_MODEL,
    'SPLIT': SPLIT,
    'MAX_ROWS': MAX_ROWS,
    'MAX_T2M_TASKS': MAX_T2M_TASKS,
    'MAX_CAPTION_TASKS': MAX_CAPTION_TASKS,
    'RUN_VALIDATION': RUN_VALIDATION,
    'HF_TOKEN_PRESENT': bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')),
})

In [ ]:
cmd = [
    sys.executable, str(BUNDLE_ROOT / 'scripts' / 'validate_gemma_soke_autoreg.py'),
    '--code-root', str(CODE_ROOT),
    '--instructions-root', str(INSTRUCTIONS_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--adapter', str(ADAPTER),
    '--base-model', BASE_MODEL,
    '--split', SPLIT,
    '--max-rows', str(MAX_ROWS),
    '--max-t2m-tasks', str(MAX_T2M_TASKS),
    '--max-caption-tasks', str(MAX_CAPTION_TASKS),
    '--max-new-motion-tokens', str(MAX_NEW_MOTION_TOKENS),
    '--max-new-text-tokens', str(MAX_NEW_TEXT_TOKENS),
    '--dtype', DTYPE,
    '--attn-implementation', ATTN_IMPLEMENTATION,
    '--norm-source', NORM_SOURCE,
    '--body-model-root', str(BODY_MODEL_ROOT),
    '--progress-every', '5',
]
if FORCE:
    cmd.append('--force')
print(' '.join(shlex.quote(str(x)) for x in cmd))
if RUN_VALIDATION:
    subprocess.run(cmd, check=True)
else:
    print('Validation skipped because SOKE_GEMMA_RUN_VALIDATION=0')

In [ ]:
import pandas as pd
from IPython.display import display

run_name = f'{SPLIT}_autoreg' + (f'_limit{MAX_ROWS}' if MAX_ROWS > 0 else '')
out_dir = OUTPUT_ROOT / run_name
table_csv = out_dir / 'autoreg_validation_compact_table.csv'
summary_csv = out_dir / 'autoreg_validation_summary_by_task_dataset.csv'
details_csv = out_dir / 'autoreg_validation_rows.csv'
table_png = out_dir / 'autoreg_validation_compact_table.png'

print('Details CSV:', details_csv)
print('Summary CSV:', summary_csv)
print('Compact table CSV:', table_csv)
print('Compact table PNG:', table_png)
if table_csv.exists():
    display(pd.read_csv(table_csv))
else:
    print('Compact table CSV is not present yet.')

In [ ]:
# Optional Colab runtime close. Set SOKE_GEMMA_CLOSE_RUNTIME=1 when running unattended.
if os.environ.get('SOKE_GEMMA_CLOSE_RUNTIME', '0') == '1':
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print('Runtime close skipped:', repr(exc))